In [ ]:
# Import modules

# Sci computing
import numpy as np
import scipy as sp
import seawater as sw
import scipy.sparse.linalg as sla
from contourpy import contour_generator
import pyfftw 

# Importing bathymetry (tiff)
import rasterio as geo

# Parallel comupting
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# For Data
import netCDF4 as nc
import xarray as xr

# Plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as grdspc
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
import cmocean as cm

# Gen stuff
from datetime import date
today = date.today()

In [ ]:
# For plotting; have the bathymetry of the gulf

ds_gulf = geo.open('/home/justin_cooke_uri_edu/DeepCyclones/gulf.tiff')
#ds_gulf = geo.open('../gulf.tiff')
img = ds_gulf.read(1)

(gm,gn) = np.shape(img)
gulf_lon = np.linspace(-90,-82,gn)
gulf_lat = np.linspace(22,28,gm)

# To get the aspect ratio right
ymid = np.mean(gulf_lat)
ymid_rad = ymid*np.pi/180

In [ ]:
ds_eref = xr.open_mfdataset('./hycom_data/hycom_etaref_*.nc',combine='nested',concat_dim='MT')
ds_ssh = xr.load_dataset('./hycom_data/hycom_ssh_filtered.nc')

In [ ]:
# Load the variables we care about
eref = ds_eref['eta_ref']
ssh = ds_ssh['ssh']
ssh_demean = ssh - xr.DataArray.mean(ssh,dim=["Latitude","Longitude"],skipna=True)

lon = ds_eref['Longitude']
lat = ds_eref['Latitude']

Nt,Nlat,Nlon = eref.shape

**FUNCTIONS**

In [ ]:
# CEOF Function

def c_eof(D,NOE=10):
    # This function determines the complex empirical orthogonal functions of a data set contained in matrix D

    # INPUTS:
    # D: Each row is assumed to be a sample; each column a variable. Thus, a column represents a time-series of one variable (or at one point)
    # NOE: Number of Eigenvalues (optional, if non-given then NOE=10)

    # OUTPUTS:
    # V: Vector of real eigenvalues 
    # EOFs: Matrix with complex values, each column represents an EOF
    # EC: EOF coefficients, also called principal component coefficients, i.e., the original time-series transformed to EOF space
    # error: The L2-norm of the reconstruction error of each point

    # Written by Justin Cooke, 2025, based on the MATLAB code written by Martijn Hooimeijer, 1999.

    # n = number of spatial points, m = number of time steps
    (n,m) = np.shape(D)
    q = np.min((n,m))

    # Hilbert transform the D matrix 
    def hilbtrans(X):

        # Now we will enter the frequency domain
        # Conduct the fft, using pyfftw so that we use an fftw library wrapper
        Y = pyfftw.interfaces.scipy_fft.fft(X,axis=0) # Need to set the axis to be zero to go along the first axis

        # Get the shape 
        (p,q) = np.shape(Y)
        N = p 
        # print('N = ',N)
        N2 = np.floor(N/2) - 1 # effect of odd and even # of elements
        N2 = np.array(np.floor(N/2),dtype=np.int16)
        # N2 = int(N2)
        # print('N2 = ',N2)

        # Now, rotate the first half of the matrix
        #print('Rotating the first half of the matrix CCW 90 degrees ...')
        P = np.zeros((p,q),dtype=complex) # Allocate a complex matrix, P, to be filled
        P[0:(N2),:] = Y[0:(N2),:]*1j
        #print('Here is the first half of P: ','\n',P[0:(N2),:])

        N2 = N - N2
        #print('New N2 = ',N2)

        P[N2:N,:] = Y[N2:N,:]/1j
        #print('Second half of P = ','\n',P[N2:N])

        # Now we need to go back to the time domain using ifft
        XH = pyfftw.interfaces.scipy_fft.ifft(P,axis=0,norm='backward')

        # XH will most likely return something that has the same real part as MATLAB, but small differences in the complex part
        
        return XH
    
    # Hilbert transform of the data matrix, D
    DH = hilbtrans(D)

    # Allocate memory for a complex matrix, DC, which is the original matrix, D, plus the hilbert transform of that matrix
    DC = np.zeros((n,m),dtype='complex')
    DC = D + DH*1j

    def EOF2(D,p): # Computes the eigenvalues, EOFs, and EOF Coefficients, input is D and the number of eigenvalues (NOE)
        
        (m,n) = np.shape(D) 
        Ma = np.sum(DC,axis=0)/m # Find the average of each time-series (column)
        DS = DC - np.tile(Ma,(m,1)) # Remove this average to make each time-series have zero-mean
        q = np.min((m,n)) # What is smaller, the time series of the number of data points?

        NOE = min(q,p) # NOE will either be the smallest number b/t the size and the inputted NOE

        if m >= n:
            CE = (DS.conj().T @ DS) / (m-1)
            if np.isscalar(CE):
                (S,EOFs) = sla.eigs(CE,NOE)
            else:
                (S,EOFs) = sla.eigs(A=CE,k=NOE,M=np.eye(n,n),return_eigenvectors=True)

        if m < n:
            CE = (DS @ DS.conj().T) / (m-1)
            (S,E) = sla.eigs(A=CE,k=NOE,M=np.eye(m,m))
            EOFs = DC.conj().T @ E
            for i in range(NOE-1):
                EOFs[:,i]  = EOFs[:,i] / np.linalg.norm(EOFs[:,i])

        V = (np.real(S))

        EC = DS @ EOFs

        diff = (DS - (EC @ EOFs.conj().T))

        error = np.sqrt(np.sum(np.abs(np.pow(diff,2)),axis=0))
        # print(error)
            
        return(V,EOFs,EC,error)

    (V,EOFs,EC,error) = EOF2(DC,NOE)

    return(V,EOFs,EC,error)

In [ ]:
# Low Pass Filter

from scipy.signal import butter, filtfilt

def lowpassfilt(data, filtT, sampT):
    """
    Low-pass filter a 1D data vector using a second-order Butterworth filter.
    Filtering is applied forward and backward to eliminate phase shift.

    Parameters
    ----------
    data : array_like
        Input data vector (1D array). Can be row or column oriented.
    filtT : float
        Filter period (same units as sampT).
    sampT : float
        Sampling period (same units as filtT).

    Returns
    -------
    out : ndarray
        Filtered data vector, same shape as input.
    """

    data = np.asarray(data)
    if data.ndim != 1:
        raise ValueError("Input must be a 1D vector, not an array.")

    # Check if input was a row vector (for shape preservation)
    was_row = data.ndim == 1 and data.shape[0] < data.shape[-1]

    # Remove linear trend between endpoints
    n = len(data)
    x = np.arange(1, n + 1)
    p = np.polyfit([1, n], [data[0], data[-1]], 1)
    lineartrend = np.polyval(p, x)
    detrended = data - lineartrend

    # Check for NaNs
    if np.isnan(detrended).any():
        raise ValueError("Data contains NaNs; remove them before filtering.")

    # Butterworth filter setup
    # Wn is the normalized cutoff frequency (Nyquist = 0.5 / sampT)
    Wn = (2.0 / filtT) * sampT
    if Wn >= 1:
        raise ValueError("Cutoff frequency too high; ensure filtT > 2 * sampT.")
    b, a = butter(2, Wn)

    # Zero-phase filtering
    filtered = filtfilt(b, a, detrended)

    # Add linear trend back
    out = filtered + lineartrend

    # Preserve input orientation
    if was_row:
        out = out.reshape(1, -1)
    else:
        out = out.reshape(-1)

    return out


**Calculate LC Characteristics**

Here, we are finding the loop current length, northern extension, and westward extension as a function of time

In [ ]:
RadEarth = 6371  # [km]

LC = {
    'length': np.zeros(Nt),
    'west': np.zeros(Nt),
    'north': np.zeros(Nt)
}

for j in range(Nt):

    z_field = ssh_demean.isel(MT=j)

    # Create a ContourGenerator instance once (x, y are 1D coordinate arrays)
    contour_gen = contour_generator(
        x=lon, y=lat, z=z_field,  # z will be passed per iteration
        line_type="Separate"          # ensures each contour is a separate array
    )


    # Generate contours for the j-th SSH slice at level 0.17
    contour_segments = contour_gen.lines(0.17)  # list of Nx2 arrays (x, y)

    curveLength = np.zeros(len(contour_segments))
    curveWest = np.zeros(len(contour_segments))
    curveNorth = np.zeros(len(contour_segments))

    for k, seg in enumerate(contour_segments):
        myX = seg[:,0]
        myY = seg[:,1]

        YCheck = myY <= 22.1
        XCheck = myX >= -86.5

        if (-83 in myX) or (np.any(YCheck) and np.any(XCheck)):
            idx_candidates = np.where(myY >= 23)[0]
            idx_w = idx_candidates[-1] if len(idx_candidates) > 0 else None

            curveNorth[k] = np.max(myY)

            if idx_w is None or np.abs(myY[idx_w]) >= 89.9:
                curveWest[k] = 0
            else:
                curveWest[k] = myX[idx_w]

            # Compute great-circle distance along the contour
            this_curve = 0.0
            for i in range(len(myX) - 1):
                phi_1 = np.deg2rad(myX[i])
                phi_2 = np.deg2rad(myX[i + 1])
                lam_1 = np.deg2rad(myY[i])
                lam_2 = np.deg2rad(myY[i + 1])

                delphi = phi_2 - phi_1
                dellam = lam_2 - lam_1

                thisDist = 2 * RadEarth * np.asin( np.sqrt(( 1 - np.cos((delphi)) +
                                                  ((np.cos(phi_1)) * np.cos(phi_2) * 
                                                   (1 - np.cos(dellam) ) )) / 2)) 

                this_curve += thisDist

            curveLength[k] = this_curve

    LC['west'][j] = np.max(np.abs(curveWest)) if len(curveWest) > 0 else 0
    LC['north'][j] = np.max(np.abs(curveNorth)) if len(curveNorth) > 0 else 0
    LC['length'][j] = np.sum(curveLength) if len(curveLength) > 0 else 0


In [ ]:
# Create time-series of eta ref averaged in the DSC region

# Create a bounding box
dsc_minlat = lat.values[40]
dsc_maxlat = lat.values[81]

dsc_minlon = lon.values[94]
dsc_maxlon = lon.values[-26]

# Select the values within the DSC bounding box and then avg them spatially
dsc_eref = eref.sel(Latitude=slice(dsc_minlat,dsc_maxlat),Longitude=slice(dsc_minlon,dsc_maxlon))
dsc_eref_mean = dsc_eref.mean(dim=['Latitude','Longitude'])

In [ ]:
# Create the D Matrix

# Get eta ref values
eref_vals = eref.values

# Create a mask containing all the points that are not NaNs
mask = ~np.isnan(eref_vals[0,:,:])
idx = np.where(mask)

# Create D
D = eref_vals[:,idx[0],idx[1]]
D = np.moveaxis(D,0,1)

lp_D = np.empty_like(D)

fc = 20.0
sc = 1.0

for i in range(D.shape[0]):
    lp_D[i,:] = lowpassfilt(D[i,:],fc,sc)

In [ ]:
# Do the CEOF

NOE=30

[V,EOFs,EC,_] = c_eof(np.transpose(lp_D),NOE)

In [ ]:
max_mode = 3

mode_key = ['M1','M2','M3']

z1 = {}
zt = {}
norm_amp = {}
phase = {}

for n,mode in enumerate(mode_key):
    factor = 1 / np.max(np.abs(EOFs[:,n]))
    z1[mode] = np.abs(EOFs[:,n]) * factor
    zt[mode] = np.abs(EC[:,n]) / factor

# Reconstruct maps in lat lon space

for n,mode in enumerate(mode_key):
    shell_amp = np.full((Nlat,Nlon),np.nan)
    shell_phase = np.full((Nlat,Nlon),np.nan)

    tempvar = EOFs[:,n]
    tempcoeff = EC[:,n]
    tempvar = 180 - np.angle(tempvar) * 180 / np.pi

    shell_amp[idx[0],idx[1]] = z1[mode]
    norm_amp[mode] = shell_amp

    shell_phase[idx[0],idx[1]] = tempvar
    phase[mode] = shell_phase

phase_masked = {}

for n,mode in enumerate(mode_key):
    
    phase_masked[mode] = np.ma.masked_where(norm_amp[mode] <= 0.2,phase[mode])


In [ ]:
# Percent for each mode
percentvar = np.empty(len(mode_key))

for n,_ in enumerate(mode_key):
    percentvar[n] = V[n] / np.sum(V)
    percentvar[n] = np.round(percentvar[n],3) * 100

plot_pervar = np.empty(NOE)

for n in range(NOE):
    plot_pervar[n] = V[n] / np.sum(V)
    plot_pervar[n] = np.round(plot_pervar[n],2) * 100

In [ ]:
plt.plot(np.linspace(1,NOE,30),plot_pervar,marker='o',linestyle='none')
plt.xlabel('Mode',fontsize=16)
plt.ylabel('$\%$ Variance',fontsize=16)
plt.title('Modes of Deep Mesoscale Eddies \nIn the Eastern Gulf',fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=14)

plt.text(2,plot_pervar[0]-0.75,f'Mode 1: {plot_pervar[0]}$\%$',fontsize=12)
plt.text(3,plot_pervar[1]-0.75,f'Mode 2: {plot_pervar[1]}$\%$',fontsize=12)
plt.text(4,plot_pervar[2]-0.75,f'Mode 3: {plot_pervar[2]}$\%$',fontsize=12)

In [ ]:
#-------------------------------------------------- PLOT AMP + PHASE + COEFFS
num_modes = 3

fig = plt.figure(figsize=(20,12))
gs0 = fig.add_gridspec(num_modes,1)
gs1 = grdspc.GridSpecFromSubplotSpec(1,num_modes,subplot_spec=gs0[0])
gs2 = grdspc.GridSpecFromSubplotSpec(1,num_modes,subplot_spec=gs0[1])
gs3 = grdspc.GridSpecFromSubplotSpec(1,1,subplot_spec=gs0[2])


for row in range(1):
    for col in range(num_modes):
        ax = fig.add_subplot(gs1[row,col])
    
        cb1 = ax.contourf(lon,lat,norm_amp[mode_key[col]],cmap=cm.cm.ice_r,levels=np.linspace(0,1.0,11),extend='both')
        ax.contour(gulf_lon,gulf_lat,np.flipud(img),colors='black',linestyles='-',levels=[-3500,-3000,-2500,-2000])
        ax.set_aspect(np.cos(ymid))
        ax.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
        ax.set_ylabel('Latitude [$^\circ$N]',fontsize=14)
        fig.colorbar(cb1,shrink=0.75)
        ax.set_title(f'{mode_key[col]}: {plot_pervar[col]}$\%$',fontsize=18)


       
for row in range(1):
    for col in range(num_modes):
        ax = fig.add_subplot(gs2[row,col])
        cb2 = ax.contourf(lon,lat,phase_masked[mode_key[col]],cmap=cm.cm.phase,levels=np.linspace(0,360,9),extend='both')
        ax.contour(gulf_lon,gulf_lat,np.flipud(img),colors='black',linestyles='-',levels=[-3500,-3000,-2500,-2000])
        ax.set_aspect(np.cos(ymid))
        ax.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
        ax.set_ylabel('Latitude [$^\circ$N]',fontsize=14)
        fig.colorbar(cb2,shrink=0.75)

mode_colors = ['xkcd:muted blue','xkcd:rusty orange','xkcd:marigold']

ax = fig.add_subplot(gs3[0])
for n,mode in enumerate(mode_key):
    if n<3:
        ax.plot(zt[mode],color=mode_colors[n],label=f'{mode}: {plot_pervar[n]}$\%$')
ax.set_xlim(0,Nt)

yr_ticks = [i for i in range(0,Nt,365)]
yr_labels = [i for i in range(0,19,1)]

ax.set_xticks(yr_ticks)
ax.set_xticklabels(yr_labels)

ax.legend(ncols=4,loc='upper center')


In [ ]:
# Conditionally average the modes 
# Find the time periods where each Mode peaks 

days = np.arange(Nt)

cond_avg = {}

for n,mode in enumerate(mode_key):
    cond_avg[mode] = {}
    cond_avg[mode]['ts'] = zt[mode] # coefficient time-series
    cond_avg[mode]['mean'] = np.mean(zt[mode]) # mean of the coefficient time-series
    cond_avg[mode]['std'] = np.std(zt[mode]) # standard dev. of the coefficient time-series
    th_mask = (zt[mode] >= (cond_avg[mode]['mean'] + cond_avg[mode]['std'] )) # make a mask where the time-series is greater than the mean plus one standard deviation
    idx_th = np.where(th_mask) # indices of the mask
    cond_avg[mode]['th'] = days[idx_th] # recorded as the threshold days
    cond_avg[mode]['length'] = LC['length'][idx_th] # LC length only using the days that meet the threshold
    cond_avg[mode]['north'] = LC['north'][idx_th] # LC northern extension only using the days that meet the threshold
    cond_avg[mode]['eref'] = eref.values[idx_th,:,:] # Eta Ref values only using the days that meet the threshold



In [ ]:
# now we need the indices for the time-series chunks
# initialize
for mode in mode_key:
    keep_me = []

    for n in range(len(cond_avg[mode]['th'])-1):
        pt_a = cond_avg[mode]['th'][n+1]
        pt_b = cond_avg[mode]['th'][n]

        diff_ab = pt_a - pt_b

        if diff_ab != 1:
            keep_me.append(n + 1)
            
    
    
    cond_avg[mode]['idx']  = np.array(keep_me)

In [ ]:
# Find LC separation times not counting detachments


sep_pt_400 = []

for n in range(Nt-1):
    pt1 = LC['length'][n]
    pt2 = LC['length'][n+1]

    pt_diff = pt1 - pt2

    if pt_diff >= 200:
        j = 1
        while j <= 40:
            pt_diff_test = pt1 - LC['length'][n+j+1]
            if pt_diff_test <= 400:
                break
            else:
                j += 1
                continue

        if j > 40:
            sep_pt_400.append(n)
            

when_separation = np.array(sep_pt_400)

In [ ]:
# Plot the LC Length with bars for the peak times of each coefficient

this_cols = 4

fig = plt.figure(figsize=(30,5*this_cols))
gs = grdspc.GridSpec(this_cols,1)

# Plotting colors
color_box = ['xkcd:muted blue','xkcd:rusty orange','xkcd:marigold']

for n in range(this_cols):
    ax = fig.add_subplot(gs[n])

    if n == 0:
        plot_me = LC['length']
        ylim_min = 250
        ylim_max = 2000
        where_box = np.array((1900,1800,1700))
        my_ylabel = 'LC Length [km]'
        rec_height = 100
        m1_txt = np.array((250,1920))
        m2_txt = np.array((250,1820))
        m3_txt = np.array((250,1720))
    elif n == 1:
        plot_me = LC['north']
        ylim_min = 23
        ylim_max = 33
        where_box = np.array((32.25,31.5,30.75))
        my_ylabel = 'LC North Ext [$^\circ$N]'
        rec_height = 0.7
        m1_txt = np.array((250,32.4))
        m2_txt = np.array((250,31.7))
        m3_txt = np.array((250,31.0))
    elif n == 2:
        plot_me = LC['west']
        ylim_min = 84.5
        ylim_max = 91.75
        where_box = np.array((91.25,90.75,90.25))
        my_ylabel = 'LC West Ext [$^\circ$W]'
        rec_height = 0.5
        m1_txt = np.array((250,91.35))
        m2_txt = np.array((250,90.85))
        m3_txt = np.array((250,90.35))
    elif n == 3:
        plot_me = dsc_eref_mean.values
        ylim_min = -0.06
        ylim_max = 0.06
        where_box = np.array((0.052,0.044,0.036))
        my_ylabel = '$\eta_{ref}$ in DSC [m]'
        rec_height = 0.008
        m1_txt = np.array((250,0.054))
        m2_txt = np.array((250,0.046))
        m3_txt = np.array((250,0.038))


    ax.plot(days,plot_me,color='k',linewidth=2)
    ax.set_ylim(ylim_min, ylim_max)

    ax.set_xticks(yr_ticks)
    ax.set_xticklabels([])
    ax.set_ylabel(my_ylabel,fontsize=16)
    ax.tick_params(axis='both', which='major', labelsize=14)

    for k,rec_mode in enumerate(mode_key):
        here_box = where_box[k]
        for i in range(len(cond_avg[rec_mode]['idx'])-1):

            if i == 0:
                pt1 = cond_avg[rec_mode]['th'][0]
                pt2 = cond_avg[rec_mode]['th'][cond_avg[rec_mode]['idx'][i]-1]
                rec_width = pt2-pt1

            else:
                pt1 = cond_avg[rec_mode]['th'][cond_avg[rec_mode]['idx'][i-1]]
                pt2 = cond_avg[rec_mode]['th'][cond_avg[rec_mode]['idx'][i]-1]
                rec_width = pt2-pt1

            ax.add_patch(Rectangle((pt1,here_box),width=rec_width,height=rec_height,color=color_box[k]))

    for nn in range(len(when_separation)):
        ax.vlines(when_separation[nn],-100,2000,color='xkcd:electric purple')
        ax.vlines(when_separation[nn]-60,-100,2000,color='xkcd:kelly green')
    ax.set_xlim(0,Nt)

    ax.text(Nt-m1_txt[0],m1_txt[1],'Mode 1',fontsize=16)
    ax.text(Nt-m2_txt[0],m2_txt[1],'Mode 2',fontsize=16)
    ax.text(Nt-m3_txt[0],m3_txt[1],'Mode 3',fontsize=16)




In [ ]:
# First determine if a mode peaks during a separation (60 days prior to separation date)

# function to see if an array has any values within a range
def in_range_of(arr,low_bound,up_bound):
    return any(low_bound <= item <=up_bound for item in arr)

# function to count the number of days the mode peaks in the range
def sum_range_of(arr,low_bound,up_bound):
    return sum(low_bound <= item <=up_bound for item in arr)


mode_max_days = np.empty((len(mode_key),len(when_separation)))
# Loops through the modes to return a true or false for modes peaking during a separation event
for m,mode in enumerate(mode_key):
    max_in_separate = []
    
    for n in range(len(when_separation)):
        sep_start = when_separation[n] - 60
        sep_end = when_separation[n] 
        my_cond = in_range_of(cond_avg[mode]['th'],sep_start,sep_end)
        max_in_separate.append(my_cond)
        mode_max_days[m,n] = sum_range_of(cond_avg[mode]['th'],sep_start,sep_end)
        
    cond_avg[mode]['in range'] = max_in_separate

dom_mode = []
for n in range(len(when_separation)):
    this_slice = mode_max_days[:,n] 
    if sum(this_slice) == 0: 
        dom_mode.append('Skip Me')
    
    dom_mode_idx = np.where(this_slice == np.max(this_slice))
    dom_mode.append(mode_key[dom_mode_idx[0][0]])

print(cond_avg['M1']['in range'])
print(cond_avg['M2']['in range'])
print(cond_avg['M3']['in range'])

In [ ]:
# LC North Extension versus DSC

fig,ax = plt.subplots(figsize=(10,12))

dsc_plot = dsc_eref_mean.values[when_separation+1]
lc_plot = LC['north'][when_separation+1]

for n in range(len(when_separation)):
    #if np.abs(dsc_plot[n]) >= 0.025:
    #    ax.plot(dsc_plot[n],lc_plot[n],marker='o',markeredgecolor='black',markerfacecolor='xkcd:kelly green')
    #else:
    #    ax.plot(dsc_plot[n],lc_plot[n],marker='o',markeredgecolor='black',markerfacecolor='xkcd:coral')
    
    this_mode = dom_mode[n]

    if this_mode == 'M1':
        m_color = 0
    elif this_mode == 'M2':
        m_color = 1
    elif this_mode == 'M3':
        m_color = 2
    else:
        continue
    
    ax.plot(dsc_plot[n],lc_plot[n],marker='o',markerfacecolor=mode_colors[m_color],markeredgecolor='k')

    if cond_avg['M1']['in range'][n] == True:
        ax.plot(dsc_plot[n],lc_plot[n],marker='s',markeredgecolor=mode_colors[0],markerfacecolor='none',markersize=10)
    
    if cond_avg['M2']['in range'][n] == True:
        ax.plot(dsc_plot[n],lc_plot[n],marker='s',markeredgecolor=mode_colors[1],markerfacecolor='none',markersize=14)
    
    if cond_avg['M3']['in range'][n] == True:
        ax.plot(dsc_plot[n],lc_plot[n],marker='s',markeredgecolor=mode_colors[2],markerfacecolor='none',markersize=18)

    
ax.set_xlabel('DSC Cyclone $\eta_{ref}$ Strength',fontsize=14)
ax.set_xlim(0.02,-0.06)
ax.set_ylabel('LC Northern Extension [$^\circ$N]',fontsize=14)


ax_inset = fig.add_axes(
    [0.55,0.6,0.3,0.25]
)

for n in range(len(when_separation)):
    this_mode = dom_mode[n]

    if this_mode == 'M1':
        m_color = 0
    elif this_mode == 'M2':
        m_color = 1
    elif this_mode == 'M3':
        m_color = 2
    else:
        continue
    
    ax_inset.contour(lon,lat,ssh_demean.values[when_separation[n]+1,:,:],[0.17],linewidths=2,colors=mode_colors[m_color])
ax_inset.contour(gulf_lon,gulf_lat,np.flipud(img),[-3500,-3000,-2500,-2000],colors='k',linestyles='-')
#fig.colorbar(cb,location='bottom',shrink=0.75,label='Elevation [m]')
ax_inset.set_aspect(np.cos(ymid))
ax_inset.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
ax_inset.set_ylabel('Latitude [$^\circ$N]',fontsize=14)


**Calculating EKE from Eta Ref**

In [ ]:
# Eta ref can be related to the geostrophic velocities via
# fv = gdpsi/dx 
# -fu = gdspi/dy


grav = 9.81
cor_param = 10e-4
gf = grav/cor_param 

# this is differentiating in degrees not meters need to fix
derefdx = eref.differentiate("Longitude")
derefdy = eref.differentiate("Latitude")

u = (-1*gf) * derefdy
v = (gf) * derefdx

**Cross Correlations of Stuff**

In [ ]:
# Cross correlation between the mode coefficients and the DSC eta ref time-series

# Demean the DSC eta ref values
dsc_eref_demean = dsc_eref_mean.values - np.mean(dsc_eref_mean.values)

corr_min = np.empty(len(mode_key))
corr_lag = np.empty(len(mode_key))

for n,mode in enumerate(mode_key):
    
    # Demean the mode coefficients
    mode_coeff_demean = cond_avg[mode]['ts'] - cond_avg[mode]['mean']

    # First get the correlation, then the lag
    cond_avg[mode]['corr'] = sp.signal.correlate(dsc_eref_demean,mode_coeff_demean,mode='full',method='auto')
    cond_avg[mode]['lag']  = sp.signal.correlation_lags(len(mode_coeff_demean),len(dsc_eref_demean))

    corr_min[n] = np.min(cond_avg[mode]['corr'])
    idx_min = np.where(cond_avg[mode]['corr'] == corr_min[n])
    corr_lag[n] = cond_avg[mode]['lag'][idx_min]

In [ ]:

fig = plt.figure(figsize=(20,10))

gs = grdspc.GridSpec(3,1)

for n,mode in enumerate(mode_key):
    ax = fig.add_subplot(gs[n])
    ax.plot(cond_avg[mode]['lag'],cond_avg[mode]['corr'],color=mode_colors[n],linewidth='2')
    #this_title = 'Correlation Between DSC $\eta_{ref}$'+f' \nand \nMode {n} Coefficient'
    #ax.set_title(this_title,fontsize=16)
    ax.text(corr_lag[n]+60,corr_min[n],f'Lag = {corr_lag[n]} Days',fontsize=12)
    ax.set_ylabel('Correlation',fontsize=12)
    if n == 2:
        ax.set_xlabel('Lag',fontsize=12)
    else:
        ax.set_xticklabels([])

fig.suptitle('Correlation between $\eta_{ref}$ and Mode Coefficients',fontsize=18)

In [ ]:
LC_length_demean = LC['length']-np.mean(LC['length'])

upper_deep_corr = sp.signal.correlate(dsc_eref_demean,LC_length_demean,mode='full',method='auto')
upper_deep_lags = sp.signal.correlation_lags(len(mode_coeff_demean),len(dsc_eref_demean))

ud_corr_lag = np.max(np.abs(upper_deep_corr))
ud_corr_lag_loc = np.where(np.abs(upper_deep_corr) == ud_corr_lag)
ud_corr_idx = upper_deep_lags[ud_corr_lag_loc]

fig,ax = plt.subplots(figsize=(20,5))
ax.plot(upper_deep_lags,upper_deep_corr,color='k',linewidth=2)
ax.text(ud_corr_idx+60,-1*ud_corr_lag,f'Lag = {ud_corr_idx} Days',fontsize=12)

**Find the Spectra at Hot Spots**

In [ ]:
# First get the indices for my points below

myLats = np.array([26, 26.75, 25, 26.75,24.5],dtype=float)
myLons = np.array([-86, -86.7, -86, -87.5, -85.5],dtype=float)

lat_idx = np.argmin(np.abs(lat.values[:, None] - myLats[None, :]), axis=0)
lon_idx = np.argmin(np.abs(lon.values[:, None] - myLons[None, :]), axis=0)

N_pts = len(myLats)

from scipy.signal import welch

myseg = 256

plot_f = np.zeros((129,N_pts))
plot_psd = np.zeros((129,N_pts))

for pts in range(N_pts):
    myvals = eref_vals[:,lat_idx[pts],lon_idx[pts]]

    f,psd = welch(myvals,1,nperseg=256,noverlap=128)

    plot_f[:,pts] = f
    plot_psd[:,pts] = f*psd

# Now find the spectra at hot-spots for reconstructed modes

for pts in range(N_pts):
    if pts == 0:
        myvals = cond_avg['e']


In [ ]:
# Plot of spectra based on full eta ref

fig, ax_main = plt.subplots(figsize=(15,10))

spectra_colors = ["#0b07c4","#03bcd4","#03cd03","#C0C400","#b80000"]

for pts in range(N_pts):
    ax_main.semilogx(plot_f[:,pts],plot_psd[:,pts],color=spectra_colors[pts],label=f'Lat: {myLats[pts]} Lon: {myLons[pts]}')
ax_main.legend(ncols=1,loc='upper left')
ax_main.set_ylabel('$\omega\cdot$PSD')
ax_main.set_xlabel('$\omega$')

ax_inset = fig.add_axes(
    [0.45,0.5,0.6,0.3]
)

cb = ax_inset.contourf(gulf_lon,gulf_lat,np.flipud(img),cmap=cm.cm.deep)
#fig.colorbar(cb,location='bottom',shrink=0.75,label='Elevation [m]')
ax_inset.set_aspect(np.cos(ymid))
ax_inset.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
ax_inset.set_ylabel('Latitude [$^\circ$N]',fontsize=14)
for pts in range(N_pts):
    ax_inset.plot(myLons[pts],myLats[pts],marker='o',markerfacecolor=spectra_colors[pts],markeredgecolor='black',linestyle='none',markersize=8)

**Recreate the deep field using each of the modes at predetermined times**

In [ ]:
# First determine if a mode peaks during a separation (60 days prior to separation date)

# function to see if an array has any values within a range
def in_range_of(arr,low_bound,up_bound):
    return any(low_bound <= item <=up_bound for item in arr)

# Loops through the modes to return a true or false for modes peaking during a separation event
for m,mode in enumerate(mode_key):
    
    # Create time-series of reconstructed modes
    arr1 = EC[:,m].reshape(-1,1)
    arr2 = EOFs[:,m].reshape(-1,1)

    arr2_tr = arr2.T

    temp_recon = arr1 @ arr2_tr

    temp_real = np.real(temp_recon)

    shell_recon = np.ma.zeros((Nt,Nlat,Nlon))

    for t in range(Nt):
        this_time = temp_real[t,:]
        shell_recon[t,idx[0],idx[1]]=this_time
    
    mask_recon = shell_recon == 0.0
    masked_recon = np.ma.masked_array(shell_recon,mask=mask_recon)
    cond_avg[mode]['eref recon'] = masked_recon


In [ ]:
# Now find the spectra at hot-spots for reconstructed modes

plot_f_recon = np.zeros((129,N_pts))
plot_psd_recon = np.zeros((129,N_pts))

for pts in range(N_pts):
    if pts == 0:
        mode = 'M1'
    elif pts == 1 or pts == 2:
        mode = 'M2'
    else:
        mode = 'M3'
        
    myvals = cond_avg[mode]['eref recon'][:,lat_idx[pts],lon_idx[pts]]

    f,psd = welch(myvals,1,nperseg=256,noverlap=128)

    plot_f_recon[:,pts] = f
    plot_psd_recon[:,pts] = f*psd


In [ ]:
# Plot of spectra based on full eta ref

fig, ax_main = plt.subplots(figsize=(15,10))

for pts in range(N_pts):
    if pts == 0:
        mcolor = 0
    elif pts == 1 or pts == 2:
        mcolor = 1
    else: 
        mcolor = 2
    ax_main.semilogx(plot_f_recon[:,pts],plot_psd_recon[:,pts],color=mode_colors[mcolor],label=f'Lat: {myLats[pts]} Lon: {myLons[pts]}')
ax_main.legend(ncols=1,loc='upper left')
ax_main.set_ylabel('$\omega\cdot$PSD')
ax_main.set_xlabel('$\omega$')

ax_inset = fig.add_axes(
    [0.45,0.5,0.6,0.3]
)

cb = ax_inset.contourf(gulf_lon,gulf_lat,np.flipud(img),cmap=cm.cm.deep)
#fig.colorbar(cb,location='bottom',shrink=0.75,label='Elevation [m]')
ax_inset.set_aspect(np.cos(ymid))
ax_inset.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
ax_inset.set_ylabel('Latitude [$^\circ$N]',fontsize=14)
for pts in range(N_pts):
    if pts == 0:
        mcolor = 0
    elif pts == 1 or pts == 2:
        mcolor = 1
    else: 
        mcolor = 2
    ax_inset.plot(myLons[pts],myLats[pts],marker='o',markerfacecolor=mode_colors[mcolor],markeredgecolor='black',linestyle='none',markersize=8)

In [ ]:
for m, mode in enumerate(mode_key):
    temp_arr = np.ma.zeros((60,Nlat,Nlon,cond_avg[mode]['in range'].count(True))) # initialize array
    ct = 0

    for n in range(len(when_separation)):
        if cond_avg[mode]['in range'][n] == True:
            sep_start = when_separation[n] - 60
            sep_end = when_separation[n]

            this_eref = cond_avg[mode]['eref recon'][sep_start:sep_end,:,:]
            temp_arr[:,:,:,ct] = this_eref
            ct += 1

    mask_arr = temp_arr == 0.0
    masked_arr = np.ma.masked_array(temp_arr,mask=mask_arr)
    cond_avg[mode]['eref sep time'] = masked_arr
    cond_avg[mode]['eref sep mean'] = np.ma.masked_array.mean(masked_arr,axis=-1)

In [ ]:
fig = plt.figure(figsize=(20,16))
gs0 = fig.add_gridspec(2,1)
gs1 = grdspc.GridSpecFromSubplotSpec(1,3,subplot_spec=gs0[0])
gs2 = grdspc.GridSpecFromSubplotSpec(1,3,subplot_spec=gs0[1])

myDays = [0, 14, 29, 44, 59]

for row in range(1):
    for col in range(3):
        ax = fig.add_subplot(gs1[row,col])

        cb1 = plt.contourf(lon,lat,cond_avg['M2']['eref sep mean'][myDays[col],:,:],cmap=cm.cm.balance)
        plt.colorbar(cb1)
        ax.contour(gulf_lon,gulf_lat,np.flipud(img),colors='black',linestyles='-',levels=[-3500,-3000,-2500,-2000])
        ax.set_aspect(np.cos(ymid))
        ax.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
        ax.set_ylabel('Latitude [$^\circ$N]',fontsize=14)

for row in range(1):
    for col in range(3):
        ax = fig.add_subplot(gs2[row,col])

       
        if col <= 1:
            cb1 = plt.contourf(lon,lat,-1*cond_avg['M2']['eref sep mean'][myDays[col+3],:,:],cmap=cm.cm.balance)
            plt.colorbar(cb1)
            ax.contour(gulf_lon,gulf_lat,np.flipud(img),colors='black',linestyles='-',levels=[-3500,-3000,-2500,-2000])
            ax.set_aspect(np.cos(ymid))
            ax.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
            ax.set_ylabel('Latitude [$^\circ$N]',fontsize=14)

In [ ]:
fig = plt.figure(figsize=(24,10),layout='constrained')
gs0 = fig.add_gridspec(4,2)
gs1 = grdspc.GridSpecFromSubplotSpec(1,1,subplot_spec=gs0[0:3,0:1])
gs2 = grdspc.GridSpecFromSubplotSpec(1,1,subplot_spec=gs0[3,0:1])

levels_etaref = np.linspace(-0.1,0.1,21)

myDay = when_separation[13]+1

ax1 = fig.add_subplot(gs1[0])

contour_eref = ax1.contourf(lon,lat,eref_vals[myDay,:,:],levels=levels_etaref,cmap=cm.cm.balance,extend='both')
contour_bath = ax1.contour(gulf_lon,gulf_lat,np.flipud(img),[-3500,-3000,-2500,-2000],colors='black',linestyles='solid')
contour_ssh = ax1.contour(lon,lat,ssh_demean[myDay,:,:],[0.17],colors='red',linewidths=2)
cb = fig.colorbar(contour_eref,location='bottom',ax=ax1,label='$\eta_{ref}$ [m]',shrink=0.5)
ax1.set_xlim(-90,-83)
ax1.set_aspect(np.cos(ymid))
ax1.set_xlabel('Longitude [$^\circ$W]',fontsize=14)
ax1.set_ylabel('Latitude [$^\circ$N]',fontsize=14)
ax1.tick_params(axis='both', which='major', labelsize=14)
title=ax1.set_title(f'Ensemble Mean Deep Ref Field - Day: {myDay}',fontsize=16)

ax2 = fig.add_subplot(gs2[0])
ax2.plot(days,LC['length'],color='k',linewidth=2)
ax2.vlines(myDay,250,2000,color='red')
ax2.set_ylim(250, 2000)
ax2.set_xlim(0,Nt)
ax2.set_xticks(yr_ticks)
ax2.set_xticklabels(yr_labels)
ax2.set_xlabel('Years',fontsize=16)
ax2.set_ylabel('LC Length [km]',fontsize=16)
ax2.tick_params(axis='both', which='major', labelsize=14)

where_box = np.array((1900,1800,1700))

for n,mode in enumerate(mode_key):
    here_box = where_box[n]
    for i in range(len(cond_avg[mode]['idx'])-1):

        if i == 0:
            pt1 = cond_avg[mode]['th'][0]
            pt2 = cond_avg[mode]['th'][cond_avg[mode]['idx'][i]-1]
            rec_width = pt2-pt1

        else:
            pt1 = cond_avg[mode]['th'][cond_avg[mode]['idx'][i-1]]
            pt2 = cond_avg[mode]['th'][cond_avg[mode]['idx'][i]-1]
            rec_width = pt2-pt1

        ax2.add_patch(Rectangle((pt1,here_box),width=rec_width,height=100,color=mode_colors[n]))

for n in range(len(when_separation)):
    ax2.vlines(when_separation[n],250,2000,color='xkcd:electric purple')
ax2.set_xlim(0,Nt)

ax2.text(Nt-250,1920,'Mode 1',fontsize=8)
ax2.text(Nt-250,1820,'Mode 2',fontsize=8)
ax2.text(Nt-250,1720,'Mode 3',fontsize=8)

In [ ]:
plt.rcParams['animation.ffmpeg_path'] = '/modules/spack/packages/linux-ubuntu24.04-x86_64_v3/gcc-13.2.0/ffmpeg-7.0.2-hpqh7mjhtopager73y4223ini6rzot3o/bin/ffmpeg'

fig = plt.figure(figsize=(20,10), layout='constrained')
gs0 = fig.add_gridspec(4,2)
gs1 = grdspc.GridSpecFromSubplotSpec(1,1,subplot_spec=gs0[0:3,0:1])
gs2 = grdspc.GridSpecFromSubplotSpec(1,1,subplot_spec=gs0[3,0:1])

levels_etaref = np.linspace(-0.1, 0.1, 21)

# -------------------------------------------------------------
# AX1: Static base contours and dynamic contourf object
# -------------------------------------------------------------
ax1 = fig.add_subplot(gs1[0])

# Static bathymetry and static SSH contour interval
ax1.contour(gulf_lon, gulf_lat, np.flipud(img),
            [-3500,-3000,-2500,-2000], colors='black',linestyles='-')

cf_ssh = ax1.contour(lon, lat, ssh_demean[0,:,:],
                         [0.17], colors='red', linewidths=2)

# Create contourf artist ONCE
cf = ax1.contourf(lon, lat, eref_vals[0,:,:],
                  levels=levels_etaref, cmap=cm.cm.balance, extend='both')

# Colorbar created ONCE
cb = fig.colorbar(cf, location='bottom', ax=ax1,
                  label='$\eta_{ref}$ [m]', shrink=0.5)

ax1.set_xlim(-90, -83)
ax1.set_aspect(np.cos(ymid))
ax1.set_xlabel('Longitude [$^\circ$W]', fontsize=14)
ax1.set_ylabel('Latitude [$^\circ$N]', fontsize=14)
ax1.tick_params(axis='both', which='major', labelsize=14)

title = ax1.set_title('Ensemble Mean Deep Ref Field - Day: 0', fontsize=16)

# -------------------------------------------------------------
# AX2: Static LC line, rectangles, and dynamic vertical line
# -------------------------------------------------------------
ax2 = fig.add_subplot(gs2[0])

# Static LC length time-series
ax2.plot(days, LC['length'], color='k', linewidth=2)

# Static rectangles (regimes)
for n, mode in enumerate(mode_key):
    here_box = where_box[n]
    for i in range(len(cond_avg[mode]['idx']) - 1):
        if i == 0:
            pt1 = cond_avg[mode]['th'][0]
            pt2 = cond_avg[mode]['th'][cond_avg[mode]['idx'][i] - 1]
        else:
            pt1 = cond_avg[mode]['th'][cond_avg[mode]['idx'][i-1]]
            pt2 = cond_avg[mode]['th'][cond_avg[mode]['idx'][i] - 1]
        rec_width = pt2 - pt1
        ax2.add_patch(Rectangle((pt1, here_box), width=rec_width,
                                height=100, color=mode_colors[n]))

# Static separation lines
for n in range(len(when_separation)):
    ax2.vlines(when_separation[n], 250, 2000, color='xkcd:electric purple')

# Dynamic vertical line (updated each frame)
vline = ax2.vlines(0, 250, 2000, color='red')

ax2.set_ylim(250, 2000)
ax2.set_xlim(0, Nt)
ax2.set_xticks(yr_ticks)
ax2.set_xticklabels(yr_labels)
ax2.set_xlabel('Years', fontsize=16)
ax2.set_ylabel('LC Length [km]', fontsize=16)
ax2.tick_params(axis='both', which='major', labelsize=14)

ax2.text(Nt-250, 1920, 'Mode 1', fontsize=8)
ax2.text(Nt-250, 1820, 'Mode 2', fontsize=8)
ax2.text(Nt-250, 1720, 'Mode 3', fontsize=8)

# -------------------------------------------------------------
# UPDATE FUNCTION (no clearing, no recreation)
# -------------------------------------------------------------
def remove_contourf(contour_obj):
    """
    Remove contourf artists robustly for Matplotlib >= 3.8 and older versions.
    """
    if hasattr(contour_obj, 'collections'):
        # Old Matplotlib (<=3.7)
        for coll in contour_obj.collections:
            coll.remove()
    else:
        # New Matplotlib (>=3.8)
        # ContourSet inherits from Collection and is itself the artist.
        contour_obj.remove()


def update(frame):
    global cf
    global cf_ssh

    # Remove old contourf in a backend-safe way
    remove_contourf(cf)
    remove_contourf(cf_ssh)

    # Create new contourf
    cf = ax1.contourf(
        lon, lat, eref_vals[frame,:,:],
        levels=levels_etaref,
        cmap=cm.cm.balance,
        extend='both'
    )

    cf_ssh = ax1.contour(
        lon, lat, ssh_demean[frame,:,:],
        [0.17], 
        colors='red', 
        linewidths=2
    )

    # Update title
    title.set_text(f'Ensemble Mean Deep Ref Field - Day: {frame}')

    # Update vertical line
    vline.set_segments([[(frame, 250), (frame, 2000)]])

    # Return artists
    return cf.collections if hasattr(cf, "collections") else [cf, title, vline]

ani = FuncAnimation(fig, update, frames=range(1, Nt), blit=False)
ani.save(f'./Figures/Videos/UpperDeep_{today}.mp4', writer='ffmpeg')
